# QTrader Analyst Quickstart
Core research workflow: load data, EDA, feature prep, vectorized backtest, metrics.

In [ ]:
from qtrader.analyst import AnalystSession

session = AnalystSession()

In [ ]:
symbol = "BTC-USD"
timeframe = "1h"

try:
    df = session.load_ohlcv(symbol, timeframe)
except Exception:
    df = session.sample_ohlcv(symbol="AAPL", days=5)

df.head(5)

In [ ]:
df = session.make_returns(df)
df.select(["timestamp", "close", "returns"]).head(5)

In [ ]:
import polars as pl

df = df.with_columns(
    pl.when(pl.col("returns") > 0).then(1).otherwise(-1).alias("signal")
)

bt = session.run_vector_backtest(df, signal_col="signal")
bt.tail(5)

In [ ]:
metrics = session.performance_metrics(bt["equity_curve"])
metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

bt.select(["equity_curve"]).to_pandas().plot()
plt.title("Equity Curve")
plt.show()